# Analisis Espacial y Correlacion - Observatorio de Educacion

**Objetivo:** Cruzar informacion espacial con tablas de indicadores educativos, calcular un indicador promedio de asistencia escolar y analizar correlaciones.

**Fuentes espaciales:** Comunas, Tasa asistencia escolar (6-10, 11-14, 15-16 anios), Personas por hogar, Sedes educativas.

**Fuentes tabulares:** Matricula 2026, Indicadores 2026, Estudiantes por docente/equipo 2026.

In [ ]:
import os, sys, zipfile

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_URL = 'https://github.com/j0rg3c45/Observatorio_de_Educacion.git'
    REPO_DIR = '/content/Observatorio_de_Educacion'
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)
    DATA_PATH = 'data/Fuentes de datos'
else:
    DATA_PATH = '../data/Fuentes de datos'

GEO_PATH = os.path.join(DATA_PATH, 'info_geografica')
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'Ruta geo: {GEO_PATH}')

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

def fmt_num(v):
    if abs(v) >= 1_000_000: return f'{v/1_000_000:.1f}M'
    elif abs(v) >= 1_000: return f'{v/1_000:.1f}K'
    else: return f'{v:,.0f}'

def fmt_eje(ax, eje='x'):
    def _f(x, pos):
        if abs(x)>=1e6: return f'{x/1e6:.1f}M'
        elif abs(x)>=1e3: return f'{x/1e3:.0f}K'
        else: return f'{x:.0f}'
    if eje=='x': ax.xaxis.set_major_formatter(mticker.FuncFormatter(_f))
    else: ax.yaxis.set_major_formatter(mticker.FuncFormatter(_f))

print('Librerias cargadas')

---
## 1. Carga de datos espaciales

In [ ]:
# 1.1 Descomprimir ZIPs si no estan extraidos
zips = [
    ('Comunas.zip', 'comunas_extracted'),
    ('Tasa_asistencia_escolar_2016.zip', 'tasa_asistencia_extracted'),
]

# Personas por hogar (nombre con tilde)
personas_zip = [f for f in os.listdir(GEO_PATH) if 'Personas' in f and f.endswith('.zip')]
if personas_zip:
    zips.append((personas_zip[0], 'personas_hogar_extracted'))

for zip_name, extract_dir in zips:
    zip_path = os.path.join(GEO_PATH, zip_name)
    out_path = os.path.join(GEO_PATH, extract_dir)
    if os.path.exists(zip_path) and not os.path.exists(out_path):
        os.makedirs(out_path, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_path)
        print(f'Extraido: {zip_name} -> {extract_dir}')
    else:
        print(f'Ya existe o no encontrado: {zip_name}')

print('Extraccion completada')

In [ ]:
# 1.2 Cargar comunas
comunas_path = os.path.join(GEO_PATH, 'comunas_extracted', 'Comunas.geojson')
gdf_comunas = gpd.read_file(comunas_path)
print(f'Comunas cargadas: {len(gdf_comunas)} poligonos')
print(f'Columnas: {list(gdf_comunas.columns)}')
print(f'CRS: {gdf_comunas.crs}')
gdf_comunas.head()

In [ ]:
# 1.3 Cargar tasa de asistencia escolar por rangos de edad
tasa_base = os.path.join(GEO_PATH, 'tasa_asistencia_extracted', 'Tasa_asistencia_escolar_2016')

# Buscar shapefiles por rango de edad
rangos = {}
for carpeta in os.listdir(tasa_base):
    carpeta_path = os.path.join(tasa_base, carpeta)
    if os.path.isdir(carpeta_path):
        shps = [f for f in os.listdir(carpeta_path) if f.endswith('.shp')]
        if shps:
            gdf = gpd.read_file(os.path.join(carpeta_path, shps[0]))
            rango_nombre = carpeta.replace('Tasa asistencia escolar ', '').replace(' 2016', '')
            rangos[rango_nombre] = gdf
            print(f'  Cargado: {rango_nombre} -> {len(gdf)} registros, cols: {list(gdf.columns)}')

print(f'\nRangos de edad cargados: {list(rangos.keys())}')

In [ ]:
# 1.4 Cargar personas por hogar
personas_path = os.path.join(GEO_PATH, 'personas_hogar_extracted')
# Buscar shapefile dentro de subcarpetas
gdf_personas = None
for root, dirs, files in os.walk(personas_path):
    for f in files:
        if f.endswith('.shp'):
            gdf_personas = gpd.read_file(os.path.join(root, f))
            print(f'Personas por hogar cargado: {len(gdf_personas)} registros')
            print(f'Columnas: {list(gdf_personas.columns)}')
            break

if gdf_personas is not None:
    gdf_personas.head()

In [ ]:
# 1.5 Cargar sedes educativas (GeoJSON)
sedes_geojson = os.path.join(GEO_PATH, 'geojson_info_geografica')
geojson_files = [f for f in os.listdir(sedes_geojson) if f.endswith('.geojson')]
gdf_sedes = gpd.read_file(os.path.join(sedes_geojson, geojson_files[0]))
print(f'Sedes educativas cargadas: {len(gdf_sedes)} puntos')
print(f'Columnas: {list(gdf_sedes.columns)}')
print(f'CRS: {gdf_sedes.crs}')
gdf_sedes.head()

---
## 2. Calcular indicador promedio de asistencia escolar

In [ ]:
# 2.1 Explorar columnas de cada rango para identificar la columna de tasa
for rango, gdf in rangos.items():
    print(f'\n--- {rango} ---')
    print(f'Columnas: {list(gdf.columns)}')
    # Mostrar columnas numericas
    num_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
    print(f'Numericas: {num_cols}')
    if num_cols:
        print(gdf[num_cols].describe().to_string())

In [ ]:
# 2.2 Unificar tasas de asistencia por zona geografica
# Identificar columna comun de zona (comuna, barrio, etc.) y columna de tasa
# NOTA: Ajustar nombres de columnas segun lo que muestre la celda anterior

# Intentar detectar automaticamente la columna de tasa y zona
dfs_tasa = []
for rango, gdf in rangos.items():
    num_cols = gdf.select_dtypes(include=[np.number]).columns.tolist()
    # Excluir columnas que parecen IDs o coordenadas
    tasa_cols = [c for c in num_cols if 'tasa' in c.lower() or 'asist' in c.lower() or 'porcen' in c.lower()]
    if not tasa_cols:
        # Tomar la primera numerica que no sea ID
        tasa_cols = [c for c in num_cols if c.lower() not in ['id', 'objectid', 'fid', 'shape_area', 'shape_leng']]
    
    if tasa_cols:
        col_tasa = tasa_cols[0]
        # Buscar columna de zona
        zona_cols = [c for c in gdf.columns if any(x in c.lower() for x in ['comuna', 'barrio', 'nombre', 'nom'])]
        col_zona = zona_cols[0] if zona_cols else gdf.columns[0]
        
        temp = gdf[[col_zona, col_tasa]].copy()
        temp.columns = ['zona', f'tasa_{rango.replace(" ", "_")}']
        dfs_tasa.append(temp)
        print(f'{rango}: zona={col_zona}, tasa={col_tasa}')

print(f'\nDataframes de tasa generados: {len(dfs_tasa)}')

In [ ]:
# 2.3 Merge de tasas y calculo del promedio
if len(dfs_tasa) > 0:
    df_tasas = dfs_tasa[0]
    for df_t in dfs_tasa[1:]:
        df_tasas = df_tasas.merge(df_t, on='zona', how='outer')
    
    # Calcular promedio de asistencia escolar (todas las edades)
    tasa_cols = [c for c in df_tasas.columns if c.startswith('tasa_')]
    df_tasas['tasa_asistencia_promedio'] = df_tasas[tasa_cols].mean(axis=1)
    
    print(f'Tabla de tasas consolidada: {df_tasas.shape}')
    print(f'\nEstadisticas del indicador promedio:')
    print(df_tasas['tasa_asistencia_promedio'].describe())
    df_tasas.head(10)
else:
    print('No se pudieron generar dataframes de tasa. Revisar columnas manualmente.')

---
## 3. Carga de datos tabulares educativos

In [ ]:
# 3.1 Cargar matricula 2026
ruta_mat = os.path.join(DATA_PATH, 'Reporte de matr\u00edcula', '01_Matricula_2026.xlsx')
if not os.path.exists(ruta_mat):
    ruta_mat = os.path.join(DATA_PATH, 'Reporte de matricula', '01_Matricula_2026.xlsx')
df_matricula = pd.read_excel(ruta_mat)
print(f'Matricula: {df_matricula.shape[0]:,} filas x {df_matricula.shape[1]} cols')

# 3.2 Cargar indicadores 2026
ruta_ind = os.path.join(DATA_PATH, 'Indicadores de eficiencia y de cobertura', '02_Indicadores_2026.xlsx')
df_indicadores = pd.read_excel(ruta_ind)
print(f'Indicadores: {df_indicadores.shape[0]:,} filas x {df_indicadores.shape[1]} cols')

# 3.3 Cargar docentes/equipos 2026
ruta_doc = os.path.join(DATA_PATH, 'Indicadores de docentes y equipos de computo', '03_Estudiantes_por_docente_y_equipo_2026.xlsx')
df_docentes = pd.read_excel(ruta_doc)
print(f'Docentes/Equipos: {df_docentes.shape[0]:,} filas x {df_docentes.shape[1]} cols')

In [ ]:
# 3.4 Identificar columna de comuna en cada tabla
print('--- Matricula ---')
print([c for c in df_matricula.columns if any(x in c.lower() for x in ['comuna', 'com', 'zona'])])
print('\n--- Indicadores ---')
print([c for c in df_indicadores.columns if any(x in c.lower() for x in ['comuna', 'com', 'zona'])])
print('\n--- Docentes ---')
print([c for c in df_docentes.columns if any(x in c.lower() for x in ['comuna', 'com', 'zona'])])

---
## 4. Agregacion por comuna y cruce de datos

In [ ]:
# 4.1 Agregar matricula por comuna
# NOTA: Ajustar nombre de columna de comuna segun lo que muestre celda 3.4
# Detectar columna de comuna
col_comuna_mat = [c for c in df_matricula.columns if 'comuna' in c.lower()]
if col_comuna_mat:
    col_comuna_mat = col_comuna_mat[0]
    # Agregar matricula total por comuna
    num_cols_mat = df_matricula.select_dtypes(include=[np.number]).columns
    mat_comuna = df_matricula.groupby(col_comuna_mat)[num_cols_mat].sum().reset_index()
    print(f'Matricula agregada por comuna: {len(mat_comuna)} comunas')
    mat_comuna.head()
else:
    print('No se encontro columna de comuna en Matricula. Columnas disponibles:')
    print(list(df_matricula.columns))

In [ ]:
# 4.2 Agregar indicadores por comuna
col_comuna_ind = [c for c in df_indicadores.columns if 'comuna' in c.lower()]
if col_comuna_ind:
    col_comuna_ind = col_comuna_ind[0]
    num_cols_ind = df_indicadores.select_dtypes(include=[np.number]).columns
    ind_comuna = df_indicadores.groupby(col_comuna_ind)[num_cols_ind].mean().reset_index()
    print(f'Indicadores promedio por comuna: {len(ind_comuna)} comunas')
    ind_comuna.head()
else:
    print('No se encontro columna de comuna en Indicadores. Columnas:')
    print(list(df_indicadores.columns))

In [ ]:
# 4.3 Agregar docentes/equipos por comuna
col_comuna_doc = [c for c in df_docentes.columns if 'comuna' in c.lower()]
if col_comuna_doc:
    col_comuna_doc = col_comuna_doc[0]
    num_cols_doc = df_docentes.select_dtypes(include=[np.number]).columns
    doc_comuna = df_docentes.groupby(col_comuna_doc)[num_cols_doc].mean().reset_index()
    print(f'Docentes/Equipos promedio por comuna: {len(doc_comuna)} comunas')
    doc_comuna.head()
else:
    print('No se encontro columna de comuna en Docentes. Columnas:')
    print(list(df_docentes.columns))